In [0]:
# --- Column → Source Tables Mapping ---
# column                           tables
# order_number      (FK)           sales_orders, order_products, promotions, clicked_items
# customer_id       (FK)           sales_orders, sales, promotions, customers, clicked_items
# product_id / item_id             order_products, sales, clicked_items
# order_date_key    (FK)           sales_orders
# qty                              sales, order_products
# unit_price (if available)        sales, order_products
# discount                         promotions
# promo_quantity                   promotions
#
# --- How to pick the right two tables for a column ---
# 1. Identify ALL tables that contain the column (see mapping above, or run the
#    query at the bottom of this cell to search the catalog automatically).
# 2. From that list, pick the table that is the "fact" or primary source for the
#    column (e.g. sales_orders for order_number, promotions for discount).
# 3. Pick the second table based on what you need to join on – usually a shared
#    FK column from the mapping above:
#      • order_number  → joins sales_orders ↔ order_products / promotions / clicked_items
#      • customer_id   → joins sales_orders ↔ customers / sales / promotions / clicked_items
#      • product_id    → joins order_products ↔ sales / clicked_items
# 4. Prefer the pair whose shared key is a true FK (marked above) so the join
#    cardinality is clean (1:1 or 1:many).
#
# --- Auto-detect: which tables contain a given column ---
# Run this query to list every table+column in the silver schema that matches a name.
# Replace 'your_column_name' with the column you are looking for.

from pyspark.sql.functions import col

silver_tables = [
    "retaildataplatform.silver.sales_orders",
    "retaildataplatform.silver.sales",
    "retaildataplatform.silver.order_products",
    "retaildataplatform.silver.promotions",
    "retaildataplatform.silver.customers",
    "retaildataplatform.silver.clicked_items",
]




In [0]:
retaildataplatform.silver.sales_orders
retaildataplatform.silver.sales
retaildataplatform.silver.order_products
retaildataplatform.silver.promotions
retaildataplatform.silver.customers
retaildataplatform.silver.clicked_items


In [0]:
%sql
select customer_id,order_number,count(*) from retaildataplatform.silver.sales_orders group by customer_id,order_number having count(*)>1

In [0]:
%sql
-- Get the latest order for a specific order_number + customer_id
-- Sort by order_timestamp descending and keep only the first row
select * from retaildataplatform.silver.sales_orders
where order_number=317568456 and customer_id=21532231
order by order_timestamp desc
limit 1

In [0]:
%sql
select * from (
select *, row_number() over(partition by order_number order by order_timestamp desc) as rn from retaildataplatform.silver.sales_orders) where rn=1

In [0]:
sales_order_df=spark.read.table("retaildataplatform.silver.sales_orders")
order_products_df=spark.read.table("retaildataplatform.silver.order_products")
promotions_df=spark.read.table("retaildataplatform.silver.promotions")

In [0]:
joined_df=sales_order_df.join(order_products_df,sales_order_df.order_number==order_products_df.order_number,"left").join(promotions_df,(order_products_df.order_number==promotions_df.order_number) & (order_products_df.id==promotions_df.promo_product_id),"left")
display(joined_df)

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions  as F

# Rebuild join with table aliases to disambiguate duplicate order_number columns
so = sales_order_df.alias("so")
op = order_products_df.alias("op")
pm = promotions_df.alias("pm")

joined_df = (so.join(op, F.col("so.order_number") == F.col("op.order_number"), "left")
              .join(pm, (F.col("op.order_number") == F.col("pm.order_number")) &
                         (F.col("op.id") == F.col("pm.promo_product_id")), "left"))

# Deduplicate: keep only the row with the latest order_timestamp per order_number
win = Window.partitionBy(F.col("so.order_number")).orderBy(F.col("so.order_timestamp").desc())
joined_df = joined_df.withColumn("rank", F.row_number().over(win)).where(F.col("rank") == 1)

display(joined_df)


In [0]:
%sql
with joined(
    select so.order_number,so.customer_id,op.id as product_id, so.order_timestamp as order_date_key,op.price, op.qty, row_number() over(partition by so.order_number order by so.order_timestamp desc) as rank from retaildataplatform.silver.sales_orders as so left join retaildataplatform.silver.order_products as op on so.order_number=op.order_number
)

select j.order_number,j.customer_id,j.product_id,j.order_date_key,j.price,j.qty,pm.discount,pm.promo_quantity from joined as j join retaildataplatform.silver.promotions as pm on j.product_id=pm.promo_product_id where rank=1